# 11 · Pronóstico probabilístico: LightGBM cuantílico directo a 13 días

La política (R, S) necesita un **cuantil** de la demanda durante el periodo de protección (13 días), no una media. En `04`–`10`, ese cuantil se arma como pronóstico puntual + stock de seguridad empírico: `S = F + z·√(F+1)`, con z calibrado por clase × quintil de volumen.

Aquí se prueba la alternativa directa: un LightGBM por cada nivel de servicio α, con pérdida cuantílica, que predice el α-cuantil de la **suma de demanda de los 13 días siguientes** a cada revisión.

- **Datos de entrenamiento:** un registro por producto y lunes (orígenes semanales desde 2012), solo ventanas sin días censurados, todas con fin ≤ 13-jul-2015, igual que LightGBM puntual.
- **Variantes:** cuantil directo y cuantil + corrección conformal por clase (calibrada en las 13 semanas previas a la simulación).
- **Diagnóstico:** cobertura real P(demanda ≤ S) en la simulación frente al α objetivo.

In [1]:
from pathlib import Path
from datetime import timedelta
import sys
import time
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PROC = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
REP = ROOT / "reports"
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))
from features import construir_features, FEATS, CAT_FEATS
from simulate import simular_rs

# --- Supuestos de la política (R, S) ---
R, L = 7, 7               # revisión semanal, lead time de 7 días
PROT = R + L - 1          # días que cubre cada pedido: del día siguiente hasta antes del próximo arribo
N_CALIB, N_SIM = 13, 26   # semanas de calibración y de simulación
H_ANUAL = 0.25            # costo de mantener: 25% anual del precio
MARGEN = 0.30             # costo de faltante: margen perdido por unidad no vendida
WARMUP = 14               # días iniciales que no se miden
ALPHAS = [0.50, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99]

df = pl.read_parquet(PROC / "m5_CA_1.parquet")
seg = pl.read_parquet(PROC / "segmentacion_CA_1.parquet")
mascara = pl.read_parquet(PROC / "mascara_CA_1.parquet")
ids_validos = (pd.read_parquet(PROC / "cv_baselines_CA_1.parquet", columns=["unique_id"])
                 ["unique_id"].unique().tolist())

# --- Calendario (todo antes de la prueba final reservada) ---
dev_end = pd.Timestamp(df["date"].max()) - timedelta(days=28)
last_review = dev_end - timedelta(days=PROT)
reviews_sim = [last_review - timedelta(days=7 * k) for k in range(N_SIM)][::-1]
sim_start = reviews_sim[0]
last_calib = sim_start - timedelta(days=PROT + 1)
origins_calib = [last_calib - timedelta(days=7 * k) for k in range(N_CALIB)][::-1]
train_end = origins_calib[0]
origenes = pd.date_range(origins_calib[0], last_review, freq="7D")
N_WINDOWS = len(origenes)

print("Entrenamiento LightGBM hasta:", train_end.date())
print("Calibración (orígenes):      ", origins_calib[0].date(), "→", last_calib.date(), f"({N_CALIB})")
print("Simulación (revisiones):     ", sim_start.date(), "→", last_review.date(), f"({N_SIM})")
print("Último día simulado:         ", dev_end.date(), f"| orígenes totales: {N_WINDOWS}")

Entrenamiento LightGBM hasta: 2015-07-13
Calibración (orígenes):       2015-07-13 → 2015-10-05 (13)
Simulación (revisiones):      2015-10-19 → 2016-04-11 (26)
Último día simulado:          2016-04-24 | orígenes totales: 40


In [2]:
dev = (df.filter(pl.col("date") <= dev_end.date())
         .with_columns(pl.col("id").cast(pl.Utf8))
         .filter(pl.col("id").is_in(ids_validos)))

inicio = dev.group_by("id").agg(inicio=pl.col("date").min())
fuera = (mascara.filter((pl.col("date") > train_end.date())
                        & pl.col("tipo").is_in(["inactivo", "pre_lanzamiento"]))
                ["id"].unique().to_list())
universo = sorted(
    inicio.filter(pl.col("inicio") <= (train_end - timedelta(days=112)).date())
          .filter(~pl.col("id").is_in(fuera))["id"].to_list()
)
print(f"Productos simulados: {len(universo):,} de {len(ids_validos):,} "
      f"(excluidos por inactividad en el periodo: {len(fuera):,})")

d = (
    dev.filter(pl.col("id").is_in(universo))
    .join(mascara.select(["id", "date", "tipo", "excluir"]), on=["id", "date"], how="left")
    .with_columns(pl.col("excluir").fill_null(False), pl.col("tipo").fill_null("normal"),
                  ventas=pl.col("sales").cast(pl.Float64))
    .sort(["id", "date"])
    .with_columns(limpia=pl.when(pl.col("excluir")).then(pl.lit(None)).otherwise(pl.col("ventas")))
    .with_columns(
        num=pl.col("limpia").fill_null(0).rolling_sum(28).shift(1).over("id"),
        den=(~pl.col("excluir")).cast(pl.Float64).rolling_sum(28).shift(1).over("id"),
        media_prod=pl.col("limpia").mean().over("id"),
    )
    .with_columns(imp=pl.when(pl.col("den") > 0).then(pl.col("num") / pl.col("den"))
                        .otherwise(pl.col("media_prod")))
    .with_columns(demanda=pl.when(pl.col("tipo") == "agotado").then(pl.col("imp"))
                            .otherwise(pl.col("ventas")))
    # Navidad: la tienda cierra, la demanda real es 0
    .with_columns(demanda=pl.when((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25))
                            .then(0.0).otherwise(pl.col("demanda")))
)

ventana = d.filter(pl.col("date") > train_end.date())
print(f"Días-producto con demanda imputada (agotado) en calibración + simulación: "
      f"{(ventana['tipo'] == 'agotado').mean():.1%}")

Productos simulados: 2,467 de 3,041 (excluidos por inactividad en el periodo: 569)


Días-producto con demanda imputada (agotado) en calibración + simulación: 2.6%


In [3]:
# --- Dataset a nivel origen: features conocidas en el origen, objetivo = demanda de los 13 días siguientes ---
dia_origen = origenes[0].weekday()
base = (
    dev.join(mascara.select(["id", "date", "excluir"]), on=["id", "date"], how="left")
    .with_columns(pl.col("excluir").fill_null(False))
    .sort(["id", "date"])
    .with_columns(limpia=pl.when(pl.col("excluir")).then(0).otherwise(pl.col("sales")).cast(pl.Float64),
                  ok=(~pl.col("excluir")).cast(pl.Float64),
                  evento=(pl.col("event_type_1").cast(pl.Utf8) != "none").cast(pl.Float64),
                  navidad=((pl.col("date").dt.month() == 12) & (pl.col("date").dt.day() == 25)).cast(pl.Float64))
)
feats = []
for w in [7, 28, 56, 112]:          # medias de venta limpia en ventanas que terminan en el origen
    feats.append((pl.col("limpia").rolling_sum(w).over("id") / pl.col("ok").rolling_sum(w).over("id")).alias(f"m{w}"))
base = base.with_columns(feats).with_columns(
    ceros28=(pl.col("sales") == 0).cast(pl.Float64).rolling_mean(28).over("id"),
    std28=pl.col("limpia").rolling_std(28).over("id"),
    excl28=pl.col("excluir").cast(pl.Float64).rolling_mean(28).over("id"),
    anio_pasado=pl.col("sales").cast(pl.Float64).shift(364 - PROT).rolling_sum(PROT).over("id"),
    # ventana futura (o+1 … o+13): calendario y precios son conocidos de antemano
    y13=pl.col("sales").cast(pl.Float64).shift(-PROT).rolling_sum(PROT).over("id"),
    excl_fut=pl.col("excluir").cast(pl.Float64).shift(-PROT).rolling_sum(PROT).over("id"),
    snap_fut=pl.col("snap").cast(pl.Float64).shift(-PROT).rolling_sum(PROT).over("id"),
    evento_fut=pl.col("evento").shift(-PROT).rolling_sum(PROT).over("id"),
    navidad_fut=pl.col("navidad").shift(-PROT).rolling_sum(PROT).over("id"),
    precio_fut=pl.col("sell_price").shift(-PROT).rolling_mean(PROT).over("id"),
    precio_rel=pl.col("sell_price") / pl.col("sell_price").rolling_mean(112).over("id"),
    precio_rel_fut=pl.col("sell_price").shift(-PROT).rolling_mean(PROT).over("id")
                   / pl.col("sell_price").rolling_mean(112).over("id"),
    semana=pl.col("date").dt.week(), mes=pl.col("date").dt.month(),
)
X_COLS = ["m7", "m28", "m56", "m112", "ceros28", "std28", "excl28", "anio_pasado", "snap_fut", "evento_fut",
          "navidad_fut", "precio_fut", "precio_rel", "precio_rel_fut", "semana", "mes", "dept_id", "clase"]
orig = (base.filter(pl.col("date").dt.weekday() == dia_origen + 1)
        .join(seg.select(id=pl.col("id").cast(pl.Utf8), clase="sb_class"), on="id", how="left")
        .select(["id", "date", "y13", "excl_fut", *X_COLS[:-2], "dept_id", "clase"])
        .to_pandas())
orig["date"] = pd.to_datetime(orig["date"]).astype("datetime64[ns]")
for c in ["dept_id", "clase"]:
    orig[c] = orig[c].astype(str).astype("category")

entren = orig[(orig["date"] + timedelta(days=PROT) <= train_end) & (orig["excl_fut"] == 0)
              & orig["m112"].notna() & orig["y13"].notna()]
pred = orig[orig["date"].isin(origenes) & orig["id"].isin(universo)]
print(f"Entrenamiento: {len(entren):,} ventanas limpias | predicción: {len(pred):,} (= {len(origenes)} orígenes × {len(universo):,})")
assert len(pred) == len(origenes) * len(universo)

Entrenamiento: 415,810 ventanas limpias | predicción: 98,680 (= 40 orígenes × 2,467)


In [4]:
# --- Un LightGBM cuantílico por nivel de servicio ---
QPARAMS = dict(objective="quantile", n_estimators=400, learning_rate=0.05, num_leaves=63,
               min_child_samples=100, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
               n_jobs=-1, verbose=-1)
Q = {}
t0 = time.time()
for a in ALPHAS:
    mq = lgb.LGBMRegressor(alpha=a, **QPARAMS).fit(entren[X_COLS], entren["y13"])
    p = pred.assign(q=np.clip(mq.predict(pred[X_COLS]), 0, None))
    Q[a] = p.pivot(index="date", columns="id", values="q").reindex(index=origenes, columns=universo)
    print(f"α = {a:.2f}: {(time.time() - t0) / 60:.1f} min")
# Cuantiles no cruzados: se fuerza monotonía en α
arr = np.maximum.accumulate(np.stack([Q[a].to_numpy() for a in ALPHAS]), axis=0)
Q = {a: pd.DataFrame(arr[i], index=origenes, columns=universo) for i, a in enumerate(ALPHAS)}

# Control: el mismo modelo directo a 13 días, pero puntual (media), para separar el efecto
# de usar información reciente (features hasta el origen) del efecto de modelar cuantiles
mp = lgb.LGBMRegressor(objective="tweedie", tweedie_variance_power=1.1,
                       **{k: v for k, v in QPARAMS.items() if k != "objective"}).fit(entren[X_COLS], entren["y13"])
F_directo = (pred.assign(f=np.clip(mp.predict(pred[X_COLS]), 0, None))
                 .pivot(index="date", columns="id", values="f").reindex(index=origenes, columns=universo))

α = 0.50: 0.2 min


α = 0.70: 0.3 min


α = 0.80: 0.4 min


α = 0.90: 0.6 min


α = 0.95: 0.8 min


α = 0.98: 0.9 min


α = 0.99: 1.0 min


In [5]:
# --- Demanda real a 13 días y pronóstico puntual de LightGBM (leídos de 06) ---
pron = pd.read_parquet(PROC / "pronosticos_13d_CA_1.parquet")
tabla13 = {m: g.pivot(index="origen", columns="id", values="unidades").reindex(index=origenes, columns=universo)
           for m, g in pron.groupby("modelo") if m in ("real", "LGB")}
A, F_lgb = tabla13["real"], tabla13["LGB"]
calib = [o for o in origenes if o <= last_calib]
simo = [o for o in origenes if o >= sim_start]

clase = (seg.select(id=pl.col("id").cast(pl.Utf8), sb_class="sb_class").to_pandas()
            .set_index("id")["sb_class"].reindex(universo))

# Pronóstico puntual + stock empírico: F + z·√(F+1), con z por clase × quintil de volumen (como en 04/06)
N_BINS = 5

def stock_empirico(Fp):
    Fc = Fp.loc[calib]
    z = (A.loc[calib] - Fc) / np.sqrt(Fc + 1)
    zl = pd.DataFrame({"id": np.tile(universo, len(calib)), "f": Fc.to_numpy().ravel(), "z": z.to_numpy().ravel()})
    zl["clase"] = zl["id"].map(clase)
    q_clase = zl.groupby("clase")["z"].quantile(ALPHAS).unstack()
    edges = np.unique(np.quantile(zl["f"], np.linspace(0, 1, N_BINS + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    zl["bin"] = pd.cut(zl["f"], edges, labels=False).astype(int)
    tabla_q = zl.groupby(["clase", "bin"])["z"].quantile(ALPHAS).unstack()

    def s(a, idx):
        Fm = Fp.loc[idx].to_numpy()
        bins = pd.cut(Fm.ravel(), edges, labels=False).astype(int)
        claves = pd.MultiIndex.from_arrays([np.tile(clase.to_numpy(), len(idx)), bins])
        qa = tabla_q[a].reindex(claves).to_numpy().reshape(Fm.shape)
        qa = np.where(np.isnan(qa), np.tile(clase.map(q_clase[a]).to_numpy(), (len(idx), 1)), qa)
        return np.maximum(0.0, Fm + qa * np.sqrt(Fm + 1))
    return s

# Conformal por clase: corrección c tal que P(A ≤ q̂ + c) = α en la calibración
def s_conformal(a, idx):
    r = (A.loc[calib] - Q[a].loc[calib])
    rl = pd.DataFrame({"clase": np.tile(clase.to_numpy(), len(calib)), "r": r.to_numpy().ravel()})
    c = rl.groupby("clase")["r"].quantile(a)
    return np.maximum(0.0, Q[a].loc[idx].to_numpy() + clase.map(c).to_numpy())

politicas = {
    "LGB + stock empírico (actual)": stock_empirico(F_lgb),
    "LGB directo 13 d + stock empírico": stock_empirico(F_directo),
    "LGB cuantílico": lambda a, idx: Q[a].loc[idx].to_numpy(),
    "LGB cuantílico + conformal": s_conformal,
}
cobertura = pd.DataFrame({n: {a: (A.loc[simo].to_numpy() <= f(a, simo)).mean() for a in ALPHAS}
                          for n, f in politicas.items()}).rename_axis("alpha")
cobertura.to_csv(REP / "probabilistico_cobertura_CA_1.csv")
cobertura.round(3)   # ideal: igual a α

,LGB + stock empírico (actual),LGB directo 13 d + stock empírico,LGB cuantílico,LGB cuantílico + conformal
alpha,,,,
0.50,0.518,0.507,0.495,0.502
0.70,0.708,0.701,0.698,0.702
0.80,0.803,0.798,0.797,0.801
0.90,0.900,0.898,0.897,0.903
0.95,0.950,0.947,0.947,0.950
0.98,0.979,0.976,0.976,0.979
0.99,0.989,0.988,0.988,0.990


In [6]:
# --- Simulación (R, S) con cada política ---
fechas = pd.date_range(origenes[0] + timedelta(days=1), dev_end)
pdd = d.select(["id", "date", "demanda", "sell_price"]).to_pandas()
pdd["date"] = pd.to_datetime(pdd["date"]).astype("datetime64[ns]")
D_full = pdd.pivot(index="date", columns="id", values="demanda").reindex(index=fechas, columns=universo)
precio_full = pdd.pivot(index="date", columns="id", values="sell_price").reindex(index=fechas, columns=universo)
sim_fechas = pd.date_range(sim_start, dev_end)
D_sim, precio = D_full.loc[sim_fechas].to_numpy(), precio_full.loc[sim_fechas].to_numpy()
revision = sim_fechas.isin(reviews_sim)
idx_rev = np.where(revision)[0]

def metricas(res):
    s = slice(WARMUP, None)
    dem, vend, perd, inv, p = (x[s] for x in (D_sim, res["vendido"], res["perdido"], res["inventario"], precio))
    c_mant, c_falt = (inv * p * H_ANUAL / 365).sum(), (perd * p * MARGEN).sum()
    return {"fill_rate": vend.sum() / dem.sum(), "inv_valor_prom": (inv * p).sum(axis=1).mean(),
            "costo_mantener": c_mant, "costo_faltante": c_falt, "costo_total": c_mant + c_falt}

filas, por_producto = [], []
w = slice(WARMUP, None)
for n, f in politicas.items():
    for a in ALPHAS:
        S = np.zeros_like(D_sim)
        S[idx_rev] = f(a, reviews_sim)
        res = simular_rs(D_sim, S, revision, L, inv_inicial=np.ceil(S[0]))
        filas.append({"politica": n, "alpha": a, **metricas(res)})
        por_producto.append(pd.DataFrame({
            "modelo": n, "alpha": a, "id": universo,
            "inv_usd_dia": (res["inventario"][w] * precio[w]).sum(axis=0),
            "perdido_usd": (res["perdido"][w] * precio[w]).sum(axis=0),
            "vendido_u": res["vendido"][w].sum(axis=0), "demanda_u": D_sim[w].sum(axis=0)}))
tabla = pd.DataFrame(filas)

ref = pd.read_csv(REP / "ablacion_censura_CA_1.csv").query("modelo == 'LGB'")
actual = tabla[tabla["politica"] == "LGB + stock empírico (actual)"]
assert np.allclose(actual["costo_total"].to_numpy(), ref["costo_total"].to_numpy(), rtol=1e-6), "No reproduce 06"
print("Control OK: la política actual reproduce 06_ablacion_censura")

def inv_en(n, fr=0.95):
    c = tabla[tabla["politica"] == n].sort_values("fill_rate")
    return np.interp(fr, c["fill_rate"], c["inv_valor_prom"]) if c["fill_rate"].min() <= fr <= c["fill_rate"].max() else np.nan

opt = tabla.loc[tabla.groupby("politica")["costo_total"].idxmin()].set_index("politica")
resumen = pd.DataFrame({"Inv_FR95": {n: inv_en(n) for n in politicas}, "Alpha_opt": opt["alpha"],
                        "Fill_opt": opt["fill_rate"], "Costo_opt": opt["costo_total"]}).loc[list(politicas)]
base_ = resumen.iloc[0]
resumen["vs_actual_%_costo"] = (resumen["Costo_opt"] / base_["Costo_opt"] - 1) * 100
resumen["vs_actual_%_inv95"] = (resumen["Inv_FR95"] / base_["Inv_FR95"] - 1) * 100
tabla.to_csv(REP / "probabilistico_CA_1.csv", index=False)
resumen.to_csv(REP / "probabilistico_resumen_CA_1.csv")
resumen.round({"Inv_FR95": 0, "Fill_opt": 4, "Costo_opt": 0, "vs_actual_%_costo": 1, "vs_actual_%_inv95": 1})

Control OK: la política actual reproduce 06_ablacion_censura


,Inv_FR95,Alpha_opt,Fill_opt,Costo_opt,vs_actual_%_costo,vs_actual_%_inv95
LGB + stock empírico (actual),120675.0,0.98,0.9900,31977.0,0.0,0.0
LGB directo 13 d + stock empírico,112669.0,0.98,0.9870,28689.0,-10.3,-6.6
LGB cuantílico,107252.0,0.98,0.9911,27132.0,-15.2,-11.1
LGB cuantílico + conformal,108181.0,0.98,0.9915,27203.0,-14.9,-10.4


In [7]:
# --- Bootstrap de productos (como en 08): ¿la ventaja sobre los clásicos corregidos es significativa? ---
pp = pd.concat([pd.concat(por_producto),
                pd.read_parquet(PROC / "ablacion_por_producto_CA_1.parquet").query("modelo in ['TSB_imp', 'AutoETS_imp', 'TSB']")])
modelos_b = list(politicas) + ["AutoETS_imp", "TSB_imp", "TSB"]
X = {col: (pp.set_index(["modelo", "alpha", "id"])[col].unstack("id").reindex(columns=universo)
              .reindex(pd.MultiIndex.from_product([modelos_b, ALPHAS])).to_numpy()
              .reshape(len(modelos_b), len(ALPHAS), len(universo)))
     for col in ["inv_usd_dia", "perdido_usd", "vendido_u", "demanda_u"]}
N_DIAS = len(sim_fechas) - WARMUP

def costo_inv(W):
    agg = {k: np.einsum("bp,map->bma", W, x) for k, x in X.items()}
    costo = (agg["inv_usd_dia"] * H_ANUAL / 365 + agg["perdido_usd"] * MARGEN).min(axis=2)
    fill, inv = agg["vendido_u"] / agg["demanda_u"], agg["inv_usd_dia"] / N_DIAS
    inv95 = np.full(costo.shape, np.nan)
    for b in range(W.shape[0]):
        for m in range(len(modelos_b)):
            o = np.argsort(fill[b, m])
            if fill[b, m, o[0]] <= 0.95 <= fill[b, m, o[-1]]:
                inv95[b, m] = np.interp(0.95, fill[b, m, o], inv[b, m, o])
    return costo, inv95

c_full, i_full = costo_inv(np.ones((1, len(universo))))
W = np.random.default_rng(7).multinomial(len(universo), np.full(len(universo), 1 / len(universo)), size=2000).astype(float)
c_bs, i_bs = costo_inv(W)
ix = {m: i for i, m in enumerate(modelos_b)}
comparaciones = {
    "LGB cuantílico vs. AutoETS corregido": ("LGB cuantílico", "AutoETS_imp"),
    "LGB cuantílico vs. TSB corregido": ("LGB cuantílico", "TSB_imp"),
    "LGB cuantílico vs. LGB actual": ("LGB cuantílico", "LGB + stock empírico (actual)"),
    "LGB directo vs. LGB actual": ("LGB directo 13 d + stock empírico", "LGB + stock empírico (actual)"),
    "LGB cuantílico vs. LGB directo": ("LGB cuantílico", "LGB directo 13 d + stock empírico"),
    "LGB cuantílico vs. TSB con ventas": ("LGB cuantílico", "TSB"),
}
filas_b = []
for nombre, (a_, b_) in comparaciones.items():
    for met, full, bs in [("costo", c_full, c_bs), ("inv95", i_full, i_bs)]:
        cambio = (bs[:, ix[a_]] / bs[:, ix[b_]] - 1) * 100
        filas_b.append({"comparacion": nombre, "metrica": met,
                        "cambio_pct": (full[0, ix[a_]] / full[0, ix[b_]] - 1) * 100,
                        "ic95_inf": np.nanpercentile(cambio, 2.5), "ic95_sup": np.nanpercentile(cambio, 97.5),
                        "prob_ahorro": np.nanmean(cambio < 0)})
ic_prob = pd.DataFrame(filas_b)
ic_prob.to_csv(REP / "probabilistico_bootstrap_CA_1.csv", index=False)
ic_prob.round(3)

,comparacion,metrica,cambio_pct,ic95_inf,ic95_sup,prob_ahorro
0,LGB cuantílico vs. AutoETS corregido,costo,-12.954,-15.935,-10.214,1.0
1,LGB cuantílico vs. AutoETS corregido,inv95,-11.402,-14.044,-8.719,1.0
2,LGB cuantílico vs. TSB corregido,costo,-13.408,-15.910,-10.725,1.0
3,LGB cuantílico vs. TSB corregido,inv95,-12.686,-14.953,-10.333,1.0
4,LGB cuantílico vs. LGB actual,costo,-15.152,-17.461,-12.486,1.0
5,LGB cuantílico vs. LGB actual,inv95,-11.123,-14.096,-7.638,1.0
6,LGB directo vs. LGB actual,costo,-10.283,-13.071,-6.884,1.0
7,LGB directo vs. LGB actual,inv95,-6.634,-9.554,-3.165,1.0
8,LGB cuantílico vs. LGB directo,costo,-5.427,-7.985,-3.025,1.0
9,LGB cuantílico vs. LGB directo,inv95,-4.808,-5.857,-3.717,1.0


In [8]:
fig = px.line(tabla, x="inv_valor_prom", y="fill_rate", color="politica", markers=True,
              hover_data=["alpha", "costo_total"],
              labels={"inv_valor_prom": "Inventario promedio (USD)", "fill_rate": "Fill rate", "politica": "Política"},
              title="Stock de seguridad empírico vs. LightGBM cuantílico — CA_1")
fig.update_layout(width=1000, height=600, yaxis_tickformat=".0%")
fig.write_html(FIG_DIR / "12_probabilistico.html", include_plotlyjs="cdn")
try:
    fig.write_image(FIG_DIR / "12_probabilistico.png", scale=2)
except Exception as e:
    print(f"PNG no generado ({e})")
fig.show()

PNG no generado (('The browser seemed to close immediately after starting.', 'You can set the `logging.Logger` level lower to see more output.', 'You may try installing a known working copy of Chrome by running ', '`$ choreo_get_chrome`.It may be your browser auto-updated and will now work upon restart. The browser we tried to start is located at /Applications/Google Chrome.app/Contents/MacOS/Google Chrome.'))


## Conclusiones

| Política (CA_1) | Inventario a fill rate 95% | Costo óptimo | vs. política actual |
|---|---|---|---|
| LightGBM puntual (rezagos ≥ 28 d) + stock empírico — actual | $120,675 | $31,977 | — |
| LightGBM directo a 13 días + stock empírico | $112,669 | $28,689 | −10.3% costo (IC 95%: −13.1 a −6.9) |
| **LightGBM cuantílico directo** | **$107,252** | **$27,132** | **−15.2% costo (−17.5 a −12.5)** |
| LightGBM cuantílico + conformal | $108,181 | $27,203 | −14.9% costo |

- **La formulación del modelo sí importa.** El LightGBM original, con rezagos de 28 días o más, ignora las últimas semanas antes de cada pedido. Pronosticar directo la demanda de los 13 días de protección, con información hasta el origen, ahorra 10%. Modelar el cuantil en lugar de la media ahorra otro 5% (−5.4%, IC −8.0 a −3.0).
- **LightGBM cuantílico supera a los clásicos corregidos:** −13.0% de costo frente a AutoETS con demanda corregida (IC 95%: −15.9 a −10.2) y −11.4% de inventario a fill rate de 95%. Frente a TSB con ventas, el punto de partida, el ahorro acumulado es −37%.
- **Calibración:** las cuatro políticas tienen cobertura real cercana al α objetivo (p. ej., 94.7–95.0% para α = 0.95). La corrección conformal casi no cambia nada: los cuantiles de LightGBM ya salen calibrados.
- **Lectura conjunta con 06:** corregir la censura es la palanca más grande (−25% a −28% con el mismo modelo), y una mejor formulación del modelo agrega −13% sobre esa base.
- **Limitaciones:** solo CA_1; el modelo cuantílico se entrena una vez (13-jul-2015) y se entrena con ventanas sin días censurados, así que también se beneficia de la corrección.